In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS total_records
FROM retail.silver.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM retail.silver.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM retail.silver.orders;

table_name,total_records
customers,35
products,20
orders,94


In [0]:
%scala

// ==========================================
// STEP 1: CREATE CUSTOMER DIMENSION
// ==========================================

// Import required Spark functions
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window

// Read cleaned customer data from Silver layer
val customerSource = spark.table("retail.silver.customers")

// Check the source record count
println("Silver Customer Count: " + customerSource.count())

// ==========================================
// STEP 2: GENERATE SURROGATE KEY
// ==========================================

// Create a window ordered by customer_id
// row_number() generates a unique sequential number
// This becomes our customer_key
val customerWindow = Window.orderBy(col("customer_id"))

val dimCustomer = customerSource
  .withColumn(
    "customer_key",
    row_number().over(customerWindow)
  )

// ==========================================
// STEP 3: SELECT REQUIRED COLUMNS
// ==========================================

// Keep only the required customer dimension fields
val finalDimCustomer = dimCustomer.select(
  col("customer_key").cast("long"),
  col("customer_id"),
  col("customer_name"),
  col("email"),
  col("city"),
  col("signup_date")
)

// ==========================================
// STEP 4: WRITE TO GOLD LAYER
// ==========================================

// Save the customer dimension as a Delta table
finalDimCustomer.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.gold.dim_customer")

// ==========================================
// STEP 5: VERIFY GOLD TABLE
// ==========================================

val goldCustomers = spark.table("retail.gold.dim_customer")

println("Gold Customer Count: " + goldCustomers.count())

// Display table structure
goldCustomers.printSchema()

// Display sample records
goldCustomers.show(10, false)


Silver Customer Count: 35
Gold Customer Count: 35
root
-- customer_key: long (nullable = true)
-- customer_id: string (nullable = true)
-- customer_name: string (nullable = true)
-- email: string (nullable = true)
-- city: string (nullable = true)
-- signup_date: date (nullable = true)

+------------+-----------+-------------+-----------------------------+---------+-----------+
customer_key|customer_id|customer_name|email |city |signup_date|
+------------+-----------+-------------+-----------------------------+---------+-----------+
1 |CUST001 |Priya Patel |priya.patel1@mailexample.in |Mumbai |2025-06-24 |
2 |CUST002 |Sneha Gupta |sneha.gupta2@mailexample.in |Kolkata |2025-03-13 |
3 |CUST003 |Divya Singh |divya.singh3@mailexample.in |Kochi |2025-02-08 |
4 |CUST004 |Suresh Gupta |suresh.gupta4@mailexample.in |Kochi |2023-09-03 |
5 |CUST005 |Aarav Gupta |aarav.gupta5@mailexample.in |Mumbai |2023-04-24 |
6 |CUST006 |Vikram Sharma|vikram.sharma6@mailexample.in|Kochi |2023-01-12 |
7 |CUST007 |Lakshmi Rao |lakshmi.rao7@mailexample.in |Pune |2023-08-04 |
8 |CUST009 |Anita Das |anita.das9@mailexample.in |Mumbai |2025-01-10 |
9 |CUST010 |Anita Sharma |anita.sharma10@mailexample.in|Hyderabad|2023-10-22 |
10 |CUST011 |Meera Iyer |meera.iyer11@mailexample.in |Chennai |2023-01-26 |
+------------+-----------+-------------+-----------------------------+---------+-----------+
only showing top 10 rows
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
val customerSource: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]
val customerWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@1e7e1053
val dimCustomer: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 7 more fields]
val finalDimCustomer: org.apache.spark.sql.DataFrame = [customer_key: bigint, customer_id: string ... 4 more fields]
val goldCustomers: org.apache.spark.sql.DataFrame = [customer_key: bigint, customer_id: string ... 4 more fields]

In [0]:
%scala

// ==========================================
// STEP 1: CREATE PRODUCT DIMENSION
// ==========================================

// Read cleaned product data from Silver layer
val productSource = spark.table("retail.silver.products")

println("Silver Product Count: " + productSource.count())

// ==========================================
// STEP 2: GENERATE SURROGATE KEY
// ==========================================

// Generate a sequential product_key using product_id
val productWindow = Window.orderBy(col("product_id"))

val dimProduct = productSource
  .withColumn(
    "product_key",
    row_number().over(productWindow)
  )

// ==========================================
// STEP 3: SELECT REQUIRED COLUMNS
// ==========================================

// Select product dimension attributes
val finalDimProduct = dimProduct.select(
  col("product_key").cast("long"),
  col("product_id"),
  col("product_name"),
  col("category"),
  col("unit_price"),
  col("active")
)

// ==========================================
// STEP 4: WRITE TO GOLD LAYER
// ==========================================

// Save as a Delta table
finalDimProduct.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.gold.dim_product")

// ==========================================
// STEP 5: VERIFY GOLD TABLE
// ==========================================

val goldProducts = spark.table("retail.gold.dim_product")

println("Gold Product Count: " + goldProducts.count())

goldProducts.printSchema()

goldProducts.show(10, false)


Silver Product Count: 20
Gold Product Count: 20
root
-- product_key: long (nullable = true)
-- product_id: string (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- unit_price: decimal(10,2) (nullable = true)
-- active: string (nullable = true)

+-----------+----------+-----------------+-----------+----------+------+
product_key|product_id|product_name |category |unit_price|active|
+-----------+----------+-----------------+-----------+----------+------+
1 |PRD001 |Wireless Earbuds |Electronics|2499.00 |Y |
2 |PRD002 |Smart Watch |Electronics|4999.00 |Y |
3 |PRD003 |Bluetooth Speaker|Electronics|1899.00 |Y |
4 |PRD005 |Cotton T-Shirt |Apparel |599.00 |Y |
5 |PRD006 |Denim Jeans |Apparel |1799.00 |Y |
6 |PRD007 |Running Shoes |Apparel |3299.00 |Y |
7 |PRD008 |Hoodie |Apparel |1499.00 |Y |
8 |PRD010 |Cookware Set |Home |2999.00 |Y |
9 |PRD012 |Bedsheet Set |Home |1299.00 |Y |
10 |PRD013 |Basmati Rice 5kg |Grocery |649.00 |Y |
+-----------+----------+-----------------+-----------+----------+------+
only showing top 10 rows
val productSource: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 6 more fields]
val productWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@20cd8206
val dimProduct: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 7 more fields]
val finalDimProduct: org.apache.spark.sql.DataFrame = [product_key: bigint, product_id: string ... 4 more fields]
val goldProducts: org.apache.spark.sql.DataFrame = [product_key: bigint, product_id: string ... 4 more fields]

In [0]:
%scala

// ==========================================
// STEP 1: GET ORDER DATE RANGE
// ==========================================

// Read valid order data from Silver layer
val orderSource = spark.table("retail.silver.orders")

// Find the earliest and latest order dates
val dateRange = orderSource.agg(
  min(col("order_date")).alias("min_date"),
  max(col("order_date")).alias("max_date")
).first()

val startDate = dateRange.getDate(0).toString
val endDate = dateRange.getDate(1).toString

println("Start Date: " + startDate)
println("End Date: " + endDate)

// ==========================================
// STEP 2: GENERATE DATE SEQUENCE
// ==========================================

// Generate one row for every date in the range
val dateSource = spark.sql(s"""
  SELECT explode(
    sequence(
      to_date('$startDate'),
      to_date('$endDate'),
      interval 1 day
    )
  ) AS full_date
""")

// ==========================================
// STEP 3: CREATE DATE ATTRIBUTES
// ==========================================

val dimDate = dateSource
  .withColumn(
    "date_key",
    date_format(col("full_date"), "yyyyMMdd").cast("int")
  )
  .withColumn("day", dayofmonth(col("full_date")))
  .withColumn("month", month(col("full_date")))
  .withColumn("month_name", date_format(col("full_date"), "MMMM"))
  .withColumn("quarter", quarter(col("full_date")))
  .withColumn("year", year(col("full_date")))
  .withColumn("day_of_week", date_format(col("full_date"), "EEEE"))
  .withColumn(
    "is_weekend",
    dayofweek(col("full_date")).isin(1, 7)
  )

// ==========================================
// STEP 4: SELECT REQUIRED COLUMNS
// ==========================================

val finalDimDate = dimDate.select(
  col("date_key"),
  col("full_date"),
  col("day"),
  col("month"),
  col("month_name"),
  col("quarter"),
  col("year"),
  col("day_of_week"),
  col("is_weekend")
)

// ==========================================
// STEP 5: SAVE TO GOLD LAYER
// ==========================================

finalDimDate.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.gold.dim_date")

// ==========================================
// STEP 6: VERIFY GOLD TABLE
// ==========================================

val goldDates = spark.table("retail.gold.dim_date")

println("Gold Date Count: " + goldDates.count())

goldDates.printSchema()

goldDates.orderBy("full_date").show(10, false)


Start Date: 2026-09-01
End Date: 2026-11-18
Gold Date Count: 79
root
-- date_key: integer (nullable = true)
-- full_date: date (nullable = true)
-- day: integer (nullable = true)
-- month: integer (nullable = true)
-- month_name: string (nullable = true)
-- quarter: integer (nullable = true)
-- year: integer (nullable = true)
-- day_of_week: string (nullable = true)
-- is_weekend: boolean (nullable = true)

+--------+----------+---+-----+----------+-------+----+-----------+----------+
date_key|full_date |day|month|month_name|quarter|year|day_of_week|is_weekend|
+--------+----------+---+-----+----------+-------+----+-----------+----------+
20260901|2026-09-01|1 |9 |September |3 |2026|Tuesday |false |
20260902|2026-09-02|2 |9 |September |3 |2026|Wednesday |false |
20260903|2026-09-03|3 |9 |September |3 |2026|Thursday |false |
20260904|2026-09-04|4 |9 |September |3 |2026|Friday |false |
20260905|2026-09-05|5 |9 |September |3 |2026|Saturday |true |
20260906|2026-09-06|6 |9 |September |3 |2026|Sunday |true |
20260907|2026-09-07|7 |9 |September |3 |2026|Monday |false |
20260908|2026-09-08|8 |9 |September |3 |2026|Tuesday |false |
20260909|2026-09-09|9 |9 |September |3 |2026|Wednesday |false |
20260910|2026-09-10|10 |9 |September |3 |2026|Thursday |false |
+--------+----------+---+-----+----------+-------+----+-----------+----------+
only showing top 10 rows
val orderSource: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 8 more fields]
val dateRange: org.apache.spark.sql.Row = [2026-09-01,2026-11-18]
val startDate: String = 2026-09-01
val endDate: String = 2026-11-18
val dateSource: org.apache.spark.sql.DataFrame = [full_date: date]
val dimDate: org.apache.spark.sql.DataFrame = [full_date: date, date_key: int ... 7 more fields]
val finalDimDate: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]
val goldDates: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]

In [0]:
%scala

// ==========================================
// STEP 1: READ ORDERS_V2 FROM ADLS
// ==========================================

val ordersV2Path =
  "abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv"

val ordersV2Raw = spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "false")
  .load(ordersV2Path)

// ==========================================
// STEP 2: ADD BRONZE METADATA
// ==========================================

// Preserve source data and add ingestion metadata
val ordersV2Bronze = ordersV2Raw
  .withColumn("_ingestion_timestamp", current_timestamp())
  .withColumn("_source_file", input_file_name())
  .withColumn("_batch_id", lit("BATCH_003"))

// ==========================================
// STEP 3: SAVE AS BRONZE DELTA TABLE
// ==========================================

// Save the incoming dataset as a new Bronze Delta table
ordersV2Bronze.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.bronze.orders_v2")

// ==========================================
// STEP 4: VERIFY BRONZE INGESTION
// ==========================================

val bronzeOrdersV2 = spark.table("retail.bronze.orders_v2")

println("Bronze Orders V2 Count: " + bronzeOrdersV2.count())

// Display schema and sample records
bronzeOrdersV2.printSchema()

bronzeOrdersV2.show(20, false)


Bronze Orders V2 Count: 12
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: string (nullable = true)
-- order_date: string (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _batch_id: string (nullable = true)

+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+--------------------------+----------------------------------------------------------------+---------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |sales_channel|discount_amount|_ingestion_timestamp |_source_file |_batch_id|
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+--------------------------+----------------------------------------------------------------+---------+
ORD000159|CUST040 |PRD019 |4 |2026-11-17|PLACED |2026-11-17 09:43:00|STORE |150.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000160|CUST019 |PRD005 |1 |2026-11-15|PLACED |2026-11-15 14:14:00|WEB |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000161|CUST005 |PRD020 |4 |2026-11-15|PLACED |2026-11-15 08:08:00|MOBILE_APP |150.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000162|CUST019 |PRD015 |3 |2026-11-15|PLACED |2026-11-15 16:43:00|MOBILE_APP |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000163|CUST027 |PRD010 |2 |2026-11-15|PLACED |2026-11-15 12:40:00|STORE |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000164|CUST004 |PRD003 |4 |2026-11-18|PLACED |2026-11-18 17:26:00|MOBILE_APP |100.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000165|CUST001 |PRD010 |1 |2026-11-16|PLACED |2026-11-16 20:08:00|MOBILE_APP |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000166|CUST030 |PRD018 |2 |2026-11-17|PLACED |2026-11-17 11:20:00|WEB |50.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000167|CUST007 |PRD007 |4 |2026-11-15|PLACED |2026-11-15 14:04:00|MOBILE_APP |100.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000168|CUST001 |PRD012 |1 |2026-11-15|PLACED |2026-11-15 12:35:00|WEB |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000169|CUST017 |PRD003 |3 |2026-11-17|PLACED |2026-11-17 16:35:00|WEB |150.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000170|CUST015 |PRD016 |3 |2026-11-15|PLACED |2026-11-15 20:50:00|WEB |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+--------------------------+----------------------------------------------------------------+---------+

val ordersV2Path: String = abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv
val ordersV2Raw: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 7 more fields]
val ordersV2Bronze: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val bronzeOrdersV2: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: str

In [0]:
%scala

// ==========================================
// STEP 1: READ BRONZE ORDERS V2
// ==========================================

val ordersV2 = spark.table("retail.bronze.orders_v2")

// ==========================================
// STEP 2: CHECK DUPLICATE ORDER IDS
// ==========================================

println("========== DUPLICATE ORDER CHECK ==========")

val duplicateV2Orders = ordersV2
  .groupBy("order_id")
  .count()
  .filter(col("count") > 1)

duplicateV2Orders.show(false)

// ==========================================
// STEP 3: CHECK EXISTING ORDERS IN SILVER
// ==========================================

println("========== EXISTING ORDER CHECK ==========")

val silverOrders = spark.table("retail.silver.orders")

val existingOrders = ordersV2
  .join(
    silverOrders.select("order_id"),
    Seq("order_id"),
    "inner"
  )

val newOrders = ordersV2.join(
  silverOrders.select("order_id"),
  Seq("order_id"),
  "left_anti"
)

println("Orders already in Silver: " + existingOrders.count())
println("New orders: " + newOrders.count())

// ==========================================
// STEP 4: CHECK QUANTITY AND DISCOUNT
// ==========================================

println("========== NUMERIC VALIDATION ==========")

val numericChecks = ordersV2
  .withColumn("quantity_int", expr("try_cast(quantity AS INT)"))
  .withColumn(
    "discount_decimal",
    expr("try_cast(discount_amount AS DECIMAL(10,2))")
  )

numericChecks.filter(
  col("quantity_int").isNull ||
  col("quantity_int") <= 0 ||
  col("discount_decimal").isNull ||
  col("discount_decimal") < 0
).select(
  "order_id",
  "quantity",
  "discount_amount",
  "quantity_int",
  "discount_decimal"
).show(false)

// ==========================================
// STEP 5: CHECK SALES CHANNEL
// ==========================================

println("========== SALES CHANNEL VALIDATION ==========")

ordersV2.filter(
  col("sales_channel").isNull ||
  !upper(trim(col("sales_channel"))).isin(
    "WEB", "MOBILE_APP", "STORE"
  )
).select("order_id", "sales_channel").show(false)

// ==========================================
// STEP 6: CHECK CUSTOMER REFERENCES
// ==========================================

println("========== CUSTOMER REFERENCE CHECK ==========")

val validCustomerIds = spark.table("retail.silver.customers")
  .select("customer_id")

val invalidCustomers = ordersV2
  .join(validCustomerIds, Seq("customer_id"), "left_anti")

invalidCustomers.select("order_id", "customer_id").show(false)

// ==========================================
// STEP 7: CHECK PRODUCT REFERENCES
// ==========================================

println("========== PRODUCT REFERENCE CHECK ==========")

val validProductIds = spark.table("retail.silver.products")
  .select("product_id")

val invalidProducts = ordersV2
  .join(validProductIds, Seq("product_id"), "left_anti")

invalidProducts.select("order_id", "product_id").show(false)

// ==========================================
// STEP 8: SUMMARY
// ==========================================

println("========== ORDERS V2 SUMMARY ==========")
println("Total records: " + ordersV2.count())
println("Duplicate order IDs: " + duplicateV2Orders.count())
println("Existing orders: " + existingOrders.count())
println("New orders: " + newOrders.count())


========== DUPLICATE ORDER CHECK ==========
+--------+-----+
order_id|count|
+--------+-----+
+--------+-----+

========== EXISTING ORDER CHECK ==========
Orders already in Silver: 0
New orders: 12
========== NUMERIC VALIDATION ==========
+--------+--------+---------------+------------+----------------+
order_id|quantity|discount_amount|quantity_int|discount_decimal|
+--------+--------+---------------+------------+----------------+
+--------+--------+---------------+------------+----------------+

========== SALES CHANNEL VALIDATION ==========
+--------+-------------+
order_id|sales_channel|
+--------+-------------+
+--------+-------------+

========== CUSTOMER REFERENCE CHECK ==========
+---------+-----------+
order_id |customer_id|
+---------+-----------+
ORD000169|CUST017 |
+---------+-----------+

========== PRODUCT REFERENCE CHECK ==========
+---------+----------+
order_id |product_id|
+---------+----------+
ORD000170|PRD016 |
+---------+----------+

========== ORDERS V2 SUMMARY ==========
Total records: 12
Duplicate order IDs: 0
Existing orders: 0
New orders: 12
val ordersV2: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val duplicateV2Orders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, count: bigint]
val silverOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 8 more fields]
val existingOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val newOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val numericChecks: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 12 more fields]
val validCustomerIds: org.apache.spark.sql.DataFrame = [customer_id: string]
val invalidCustomers: org.apache.spark.sql.DataFrame = [customer_id: string, order_id: string ... 10 more fields]
val validProductIds: org.apache.spark.sql.DataFrame = [product_id: string]
val invalidProducts: org.apache.spark.sql.DataFrame = [product_id: string, order_id: string ... 10 more fields]

In [0]:
%scala

// ==========================================
// STEP 1: NORMALIZE ORDERS V2
// ==========================================

val normalizedOrdersV2 = ordersV2
  .withColumn("quantity", expr("try_cast(quantity AS INT)"))
  .withColumn(
    "order_date",
    coalesce(
      expr("try_to_date(trim(order_date), 'yyyy-MM-dd')"),
      expr("try_to_date(trim(order_date), 'dd-MM-yyyy')")
    )
  )
  .withColumn("order_status", upper(trim(col("order_status"))))
  .withColumn("sales_channel", upper(trim(col("sales_channel"))))
  .withColumn(
    "discount_amount",
    expr("try_cast(discount_amount AS DECIMAL(10,2))")
  )
  .withColumn(
    "last_updated_ts",
    to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss")
  )

// ==========================================
// STEP 2: PREPARE REFERENCE DATA
// ==========================================

val customerReference = spark.table("retail.silver.customers")
  .select(col("customer_id").alias("_valid_customer_id"))

val productReference = spark.table("retail.silver.products")
  .select(col("product_id").alias("_valid_product_id"))

// Join with customer and product dimensions
val ordersWithReferences = normalizedOrdersV2
  .join(
    customerReference,
    col("customer_id") === col("_valid_customer_id"),
    "left"
  )
  .join(
    productReference,
    col("product_id") === col("_valid_product_id"),
    "left"
  )

// ==========================================
// STEP 3: VALIDATE ALL BUSINESS RULES
// ==========================================

val validatedOrdersV2 = ordersWithReferences
  .withColumn(
    "rejection_reason",
    concat_ws(" | ",
      when(col("order_id").isNull || trim(col("order_id")) === "",
        lit("MISSING_ORDER_ID")),
      when(col("quantity").isNull || col("quantity") <= 0,
        lit("INVALID_QUANTITY")),
      when(col("order_date").isNull,
        lit("INVALID_ORDER_DATE")),
      when(col("order_date") > lit("2026-11-20").cast("date"),
        lit("FUTURE_ORDER_DATE")),
      when(col("order_status").isNull ||
        !col("order_status").isin(
          "PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"
        ), lit("INVALID_ORDER_STATUS")),
      when(col("last_updated_ts").isNull,
        lit("INVALID_LAST_UPDATED")),
      when(col("sales_channel").isNull ||
        !col("sales_channel").isin("WEB", "MOBILE_APP", "STORE"),
        lit("INVALID_SALES_CHANNEL")),
      when(col("discount_amount").isNull || col("discount_amount") < 0,
        lit("INVALID_DISCOUNT_AMOUNT")),
      when(col("customer_id").isNull || trim(col("customer_id")) === "",
        lit("MISSING_CUSTOMER_ID")),
      when(col("customer_id").isNotNull &&
        col("_valid_customer_id").isNull,
        lit("INVALID_CUSTOMER_REFERENCE")),
      when(col("product_id").isNull || trim(col("product_id")) === "",
        lit("MISSING_PRODUCT_ID")),
      when(col("product_id").isNotNull &&
        col("_valid_product_id").isNull,
        lit("INVALID_PRODUCT_REFERENCE"))
    )
  )
  .drop("_valid_customer_id", "_valid_product_id")

// ==========================================
// STEP 4: SPLIT VALID AND REJECTED RECORDS
// ==========================================

val validOrdersV2 = validatedOrdersV2
  .filter(col("rejection_reason") === "")
  .drop("rejection_reason", "last_updated_ts")

val rejectedOrdersV2 = validatedOrdersV2
  .filter(col("rejection_reason") =!= "")

// ==========================================
// STEP 5: DISPLAY RESULTS
// ==========================================

println("========== ORDERS V2 VALIDATION SUMMARY ==========")

println("Total records: " + ordersV2.count())
println("Valid records: " + validOrdersV2.count())
println("Rejected records: " + rejectedOrdersV2.count())

println("========== VALID ORDERS ==========")
validOrdersV2.show(false)

println("========== REJECTED ORDERS ==========")
rejectedOrdersV2.select(
  "order_id",
  "customer_id",
  "product_id",
  "rejection_reason"
).show(false)

println("========== VALID ORDERS SCHEMA ==========")
validOrdersV2.printSchema()


========== ORDERS V2 VALIDATION SUMMARY ==========
Total records: 12
Valid records: 10
Rejected records: 2
========== VALID ORDERS ==========
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+--------------------------+----------------------------------------------------------------+---------+
order_id |customer_id|product_id|quantity|order_date|order_status|last_updated |sales_channel|discount_amount|_ingestion_timestamp |_source_file |_batch_id|
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+--------------------------+----------------------------------------------------------------+---------+
ORD000159|CUST040 |PRD019 |4 |2026-11-17|PLACED |2026-11-17 09:43:00|STORE |150.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000160|CUST019 |PRD005 |1 |2026-11-15|PLACED |2026-11-15 14:14:00|WEB |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000161|CUST005 |PRD020 |4 |2026-11-15|PLACED |2026-11-15 08:08:00|MOBILE_APP |150.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000162|CUST019 |PRD015 |3 |2026-11-15|PLACED |2026-11-15 16:43:00|MOBILE_APP |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000163|CUST027 |PRD010 |2 |2026-11-15|PLACED |2026-11-15 12:40:00|STORE |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000164|CUST004 |PRD003 |4 |2026-11-18|PLACED |2026-11-18 17:26:00|MOBILE_APP |100.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000165|CUST001 |PRD010 |1 |2026-11-16|PLACED |2026-11-16 20:08:00|MOBILE_APP |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000166|CUST030 |PRD018 |2 |2026-11-17|PLACED |2026-11-17 11:20:00|WEB |50.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000167|CUST007 |PRD007 |4 |2026-11-15|PLACED |2026-11-15 14:04:00|MOBILE_APP |100.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
ORD000168|CUST001 |PRD012 |1 |2026-11-15|PLACED |2026-11-15 12:35:00|WEB |0.00 |2026-11-20 10:54:37.268159|abfss://raw@adlsacntext.dfs.core.windows.net/nexus/orders_v2.csv|BATCH_003|
+---------+-----------+----------+--------+----------+------------+-------------------+-------------+---------------+--------------------------+----------------------------------------------------------------+---------+

========== REJECTED ORDERS ==========
+---------+-----------+----------+--------------------------+
order_id |customer_id|product_id|rejection_reason |
+---------+-----------+----------+--------------------------+
ORD000169|CUST017 |PRD003 |INVALID_CUSTOMER_REFERENCE|
ORD000170|CUST015 |PRD016 |INVALID_PRODUCT_REFERENCE |
+---------+-----------+----------+--------------------------+

========== VALID ORDERS SCHEMA ==========
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- order_date: date (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _batch_id: string (nullable = true)

val normalizedOrdersV2: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 11 more fields]
val customerReference: org.apache.spark.sql.DataFrame = [_valid_customer_id: string]
val productRefere

In [0]:
%scala

// ==========================================
// STEP 1: CHECK EXISTING SILVER TABLE
// ==========================================

import io.delta.tables.DeltaTable

val silverTableName = "retail.silver.orders"

val silverCountBeforeV2 = spark.table(silverTableName).count()

println("Silver orders before schema evolution: " + silverCountBeforeV2)

// ==========================================
// STEP 2: EVOLVE SILVER TABLE SCHEMA
// ==========================================

// Check whether the new columns already exist
// This makes the schema update safe to rerun

val existingColumns = spark.table(silverTableName)
  .columns
  .toSet

// Add sales_channel if it does not exist
if (!existingColumns.contains("sales_channel")) {
  spark.sql(s"""
    ALTER TABLE $silverTableName
    ADD COLUMNS (sales_channel STRING)
  """)
}

// Add discount_amount if it does not exist
if (!existingColumns.contains("discount_amount")) {
  spark.sql(s"""
    ALTER TABLE $silverTableName
    ADD COLUMNS (discount_amount DECIMAL(10,2))
  """)
}

println("Schema evolution completed!")

// ==========================================
// STEP 3: VERIFY EVOLVED SCHEMA
// ==========================================

val evolvedSilverOrders = spark.table(silverTableName)

evolvedSilverOrders.printSchema()

// ==========================================
// STEP 4: MERGE VALID ORDERS V2
// ==========================================

// Use Delta MERGE to insert new orders.
// Existing order IDs will not be inserted again.
// This makes the operation safe to retry.

val silverOrdersDelta = DeltaTable.forName(
  spark,
  silverTableName
)

silverOrdersDelta
  .as("target")
  .merge(
    validOrdersV2.as("source"),
    "target.order_id = source.order_id"
  )
  .whenNotMatched()
  .insertAll()
  .execute()

// ==========================================
// STEP 5: VERIFY MERGE RESULTS
// ==========================================

val finalSilverOrders = spark.table(silverTableName)

println("========== FINAL SILVER ORDER SUMMARY ==========")

println("Orders before schema evolution: " + silverCountBeforeV2)
println("Orders after schema evolution: " + finalSilverOrders.count())

// Count records inserted from BATCH_003
println(
  "New V2 orders inserted: " +
  finalSilverOrders.filter(col("_batch_id") === "BATCH_003").count()
)

// Check old records where new columns are NULL
println(
  "Old records with NULL sales_channel: " +
  finalSilverOrders.filter(
    col("_batch_id") =!= "BATCH_003" &&
    col("sales_channel").isNull
  ).count()
)

// Display newly inserted records
finalSilverOrders
  .filter(col("_batch_id") === "BATCH_003")
  .select(
    "order_id",
    "customer_id",
    "product_id",
    "sales_channel",
    "discount_amount"
  )
  .show(false)


Silver orders before schema evolution: 94
Schema evolution completed!
root
-- order_id: string (nullable = true)
-- customer_id: string (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- order_date: date (nullable = true)
-- order_status: string (nullable = true)
-- last_updated: string (nullable = true)
-- _ingestion_timestamp: timestamp (nullable = true)
-- _source_file: string (nullable = true)
-- _batch_id: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)

========== FINAL SILVER ORDER SUMMARY ==========
Orders before schema evolution: 94
Orders after schema evolution: 104
New V2 orders inserted: 10
Old records with NULL sales_channel: 94
+---------+-----------+----------+-------------+---------------+
order_id |customer_id|product_id|sales_channel|discount_amount|
+---------+-----------+----------+-------------+---------------+
ORD000159|CUST040 |PRD019 |STORE |150.00 |
ORD000160|CUST019 |PRD005 |WEB |0.00 |
ORD000161|CUST005 |PRD020 |MOBILE_APP |150.00 |
ORD000162|CUST019 |PRD015 |MOBILE_APP |0.00 |
ORD000163|CUST027 |PRD010 |STORE |0.00 |
ORD000164|CUST004 |PRD003 |MOBILE_APP |100.00 |
ORD000165|CUST001 |PRD010 |MOBILE_APP |0.00 |
ORD000166|CUST030 |PRD018 |WEB |50.00 |
ORD000167|CUST007 |PRD007 |MOBILE_APP |100.00 |
ORD000168|CUST001 |PRD012 |WEB |0.00 |
+---------+-----------+----------+-------------+---------------+

import io.delta.tables.DeltaTable
val silverTableName: String = retail.silver.orders
val silverCountBeforeV2: Long = 94
val existingColumns: scala.collection.immutable.Set[String] = HashSet(customer_id, _batch_id, _source_file, order_date, product_id, _ingestion_timestamp, quantity, order_id, order_status, last_updated)
val evolvedSilverOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val silverOrdersDelta: io.delta.tables.DeltaTable = io.delta.tables.DeltaTable@6a083679
val finalSilverOrders: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]

In [0]:
%scala

// ==========================================
// STEP 1: READ SILVER ORDERS
// ==========================================

// Read cleaned and schema-evolved order data
val silverOrdersSource = spark.table("retail.silver.orders")

// Read Gold dimension tables
val customerDimension = spark.table("retail.gold.dim_customer")
val productDimension = spark.table("retail.gold.dim_product")
val dateDimension = spark.table("retail.gold.dim_date")

println("Silver Orders: " + silverOrdersSource.count())

// ==========================================
// STEP 2: JOIN CUSTOMER DIMENSION
// ==========================================

// Match source customer_id with the customer dimension
// Retrieve the warehouse-generated customer_key

val ordersWithCustomerKey = silverOrdersSource
  .join(
    customerDimension.select("customer_id", "customer_key"),
    Seq("customer_id"),
    "inner"
  )

// ==========================================
// STEP 3: JOIN PRODUCT DIMENSION
// ==========================================

// Match source product_id with product dimension
// Retrieve product_key

val ordersWithProductKey = ordersWithCustomerKey
  .join(
    productDimension.select("product_id", "product_key"),
    Seq("product_id"),
    "inner"
  )

// ==========================================
// STEP 4: JOIN DATE DIMENSION
// ==========================================

// Match order_date with full_date
// Retrieve date_key

val ordersWithDateKey = ordersWithProductKey
  .join(
    dateDimension.select(
      col("full_date").alias("_dimension_date"),
      col("date_key")
    ),
    col("order_date") === col("_dimension_date"),
    "inner"
  )

// ==========================================
// STEP 5: GENERATE SALES KEY
// ==========================================

// Generate a sequential sales_key for the initial load
val salesWindow = Window.orderBy(col("order_id"))

val factOrder = ordersWithDateKey
  .withColumn(
    "sales_key",
    row_number().over(salesWindow).cast("long")
  )

// ==========================================
// STEP 6: SELECT FACT TABLE COLUMNS
// ==========================================

val finalFactOrder = factOrder.select(
  col("sales_key"),
  col("order_id"),
  col("customer_key"),
  col("product_key"),
  col("date_key"),
  col("quantity"),
  col("order_status"),
  col("sales_channel"),
  col("discount_amount"),
  col("last_updated")
)

// ==========================================
// STEP 7: SAVE FACT TABLE IN GOLD
// ==========================================

finalFactOrder.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("retail.gold.fact_order")

// ==========================================
// STEP 8: VERIFY FACT TABLE
// ==========================================

val goldFactOrder = spark.table("retail.gold.fact_order")

println("========== FACT ORDER SUMMARY ==========")

println("Silver Orders: " + silverOrdersSource.count())
println("Gold Fact Orders: " + goldFactOrder.count())

goldFactOrder.printSchema()

goldFactOrder.show(10, false)


Silver Orders: 104
========== FACT ORDER SUMMARY ==========
Silver Orders: 104
Gold Fact Orders: 104
root
-- sales_key: long (nullable = true)
-- order_id: string (nullable = true)
-- customer_key: long (nullable = true)
-- product_key: long (nullable = true)
-- date_key: integer (nullable = true)
-- quantity: integer (nullable = true)
-- order_status: string (nullable = true)
-- sales_channel: string (nullable = true)
-- discount_amount: decimal(10,2) (nullable = true)
-- last_updated: string (nullable = true)

+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
sales_key|order_id |customer_key|product_key|date_key|quantity|order_status|sales_channel|discount_amount|last_updated |
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
1 |ORD000001|20 |4 |20260906|1 |DELIVERED |NULL |NULL |2026-09-06 20:34:17|
2 |ORD000002|28 |4 |20261114|5 |DELIVERED |NULL |NULL |2026-11-14 17:49:59|
3 |ORD000003|29 |8 |20261031|1 |DELIVERED |NULL |NULL |2026-10-31 17:08:38|
4 |ORD000007|18 |20 |20260904|3 |DELIVERED |NULL |NULL |2026-09-04 08:32:30|
5 |ORD000008|26 |16 |20261012|5 |DELIVERED |NULL |NULL |2026-10-12 22:07:31|
6 |ORD000010|5 |15 |20261116|4 |DELIVERED |NULL |NULL |2026-11-16 11:43:45|
7 |ORD000011|3 |18 |20260926|5 |DELIVERED |NULL |NULL |2026-09-29 18:58:11|
8 |ORD000014|30 |12 |20260924|1 |DELIVERED |NULL |NULL |2026-09-27 14:17:27|
9 |ORD000016|18 |14 |20261024|4 |DELIVERED |NULL |NULL |2026-10-27 12:29:11|
10 |ORD000019|10 |9 |20260924|5 |CANCELLED |NULL |NULL |2026-09-24 08:22:54|
+---------+---------+------------+-----------+--------+--------+------------+-------------+---------------+-------------------+
only showing top 10 rows
val silverOrdersSource: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val customerDimension: org.apache.spark.sql.DataFrame = [customer_key: bigint, customer_id: string ... 4 more fields]
val productDimension: org.apache.spark.sql.DataFrame = [product_key: bigint, product_id: string ... 4 more fields]
val dateDimension: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]
val ordersWithCustomerKey: org.apache.spark.sql.DataFrame = [customer_id: string, order_id: string ... 11 more fields]
val ordersWithProductKey: org.apache.spark.sql.DataFrame = [product_id: string, customer_id: string ... 12 more fields]
val ordersWithDateKey: org.apache.spark.sql.DataFrame = [product_id: string, customer_id: string ... 14 more fields]
val salesWindow: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@42aa0fdf
val factOrder: org.apache.spark.sql.DataFrame = [product_id: string, customer_id: string ... 15 more fields]
val finalFactOrder: org.apache.spark.sql.DataFrame = [sales_key: bigint, order_id: string ... 8 more fields]
val goldFactOrder: org.apache.spark.sql.DataFrame = [sales_key: bigint, order_id: string ... 8 more fields]

In [0]:
%scala

// ==========================================
// FACT ORDER DATA QUALITY CHECKS
// ==========================================

// Read Gold fact and dimension tables
val fact = spark.table("retail.gold.fact_order")

val customerDim = spark.table("retail.gold.dim_customer")
val productDim = spark.table("retail.gold.dim_product")
val dateDim = spark.table("retail.gold.dim_date")

// ==========================================
// CHECK 1: DUPLICATE ORDER IDS
// ==========================================

val duplicateFactOrders = fact
  .groupBy("order_id")
  .count()
  .filter(col("count") > 1)

println("Duplicate Order IDs: " + duplicateFactOrders.count())

// ==========================================
// CHECK 2: DUPLICATE SALES KEYS
// ==========================================

val duplicateSalesKeys = fact
  .groupBy("sales_key")
  .count()
  .filter(col("count") > 1)

println("Duplicate Sales Keys: " + duplicateSalesKeys.count())

// ==========================================
// CHECK 3: NULL FOREIGN KEYS
// ==========================================

val nullDimensionKeys = fact.filter(
  col("customer_key").isNull ||
  col("product_key").isNull ||
  col("date_key").isNull
)

println("Records with NULL Dimension Keys: " + nullDimensionKeys.count())

// ==========================================
// CHECK 4: ORPHAN CUSTOMER KEYS
// ==========================================

val orphanCustomerKeys = fact
  .select("customer_key")
  .distinct()
  .join(
    customerDim.select("customer_key"),
    Seq("customer_key"),
    "left_anti"
  )

println("Orphan Customer Keys: " + orphanCustomerKeys.count())

// ==========================================
// CHECK 5: ORPHAN PRODUCT KEYS
// ==========================================

val orphanProductKeys = fact
  .select("product_key")
  .distinct()
  .join(
    productDim.select("product_key"),
    Seq("product_key"),
    "left_anti"
  )

println("Orphan Product Keys: " + orphanProductKeys.count())

// ==========================================
// CHECK 6: ORPHAN DATE KEYS
// ==========================================

val orphanDateKeys = fact
  .select("date_key")
  .distinct()
  .join(
    dateDim.select("date_key"),
    Seq("date_key"),
    "left_anti"
  )

println("Orphan Date Keys: " + orphanDateKeys.count())

// ==========================================
// CHECK 7: SILVER VS GOLD COUNT
// ==========================================

val silverCount = spark.table("retail.silver.orders").count()
val goldCount = fact.count()

println("Silver Order Count: " + silverCount)
println("Gold Fact Order Count: " + goldCount)

println(
  "Record Count Match: " + (silverCount == goldCount)
)

// ==========================================
// FINAL SUMMARY
// ==========================================

println("========== FACT QUALITY CHECKS COMPLETED ==========")


Duplicate Order IDs: 0
Duplicate Sales Keys: 0
Records with NULL Dimension Keys: 0
Orphan Customer Keys: 0
Orphan Product Keys: 0
Orphan Date Keys: 0
Silver Order Count: 104
Gold Fact Order Count: 104
Record Count Match: true
========== FACT QUALITY CHECKS COMPLETED ==========
val fact: org.apache.spark.sql.DataFrame = [sales_key: bigint, order_id: string ... 8 more fields]
val customerDim: org.apache.spark.sql.DataFrame = [customer_key: bigint, customer_id: string ... 4 more fields]
val productDim: org.apache.spark.sql.DataFrame = [product_key: bigint, product_id: string ... 4 more fields]
val dateDim: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]
val duplicateFactOrders: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, count: bigint]
val duplicateSalesKeys: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [sales_key: bigint, count: bigint]
val nullDimensionKeys: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [sales_key: bigint, order_id: string ... 8 more fields]
val orphanCustomerKeys: org.apache.spark.sql.DataFrame = [customer_key: bigint]
val orphanProductKeys: org.apache.spark.sql.DataFrame = [product_key: bigint]
val orphanDateKeys: org.apache.spark.sql.DataFrame = [date_key: int]
val silverCount: Long = 104
val goldCount: Long = 104

In [0]:
%scala

// ==========================================
// MONTHLY ORDER TREND ANALYSIS
// ==========================================

// Read Gold fact and date dimension tables
val factOrders = spark.table("retail.gold.fact_order")
val dateDim = spark.table("retail.gold.dim_date")

// Join Fact_Order with Dim_Date
// date_key connects the two tables
val monthlyOrderData = factOrders
  .join(
    dateDim.select("date_key", "year", "month", "month_name"),
    Seq("date_key"),
    "inner"
  )

// Group orders by year and month
// Calculate order count and total quantity
val monthlyOrderTrend = monthlyOrderData
  .groupBy("year", "month", "month_name")
  .agg(
    countDistinct("order_id").alias("total_orders"),
    sum("quantity").alias("total_quantity")
  )
  .orderBy("year", "month")

// Display monthly order trend
println("========== MONTHLY ORDER TREND ==========")

monthlyOrderTrend.show(100, false)


========== MONTHLY ORDER TREND ==========
+----+-----+----------+------------+--------------+
year|month|month_name|total_orders|total_quantity|
+----+-----+----------+------------+--------------+
2026|9 |September |29 |86 |
2026|10 |October |30 |96 |
2026|11 |November |45 |136 |
+----+-----+----------+------------+--------------+

val factOrders: org.apache.spark.sql.DataFrame = [sales_key: bigint, order_id: string ... 8 more fields]
val dateDim: org.apache.spark.sql.DataFrame = [date_key: int, full_date: date ... 7 more fields]
val monthlyOrderData: org.apache.spark.sql.DataFrame = [date_key: int, sales_key: bigint ... 11 more fields]
val monthlyOrderTrend: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [year: int, month: int ... 3 more fields]